# Lab 03 — Statistical Process Control

將 Shewhart、EWMA、CUSUM 套用在網路監控特徵上，觀察不同異常型態的敏感度。

In [ ]:
from pathlib import Path
from IPython.display import SVG, display

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for base in (start, *start.parents):
        if (base / "environment.yml").exists():
            return base
    raise FileNotFoundError("Could not find project root containing environment.yml")

svg_candidates = [
    Path("diagrams/lab03_pipeline_position.svg"),
    find_project_root() / "labs/self-study" / "diagrams/lab03_pipeline_position.svg",
]
for svg_path in svg_candidates:
    if svg_path.exists():
        display(SVG(filename=str(svg_path)))
        break
else:
    raise FileNotFoundError("Could not find diagram: diagrams/lab03_pipeline_position.svg")


## Lab 03 概覽

### 學習目標

1. 理解統計製程控制（SPC）的歷史背景與 Shewhart 管制圖的 3σ 邏輯
2. 計算 control limits 並辨識 Western Electric 違規型態
3. 理解 EWMA 的指數衰減記憶與 CUSUM 的累積偏差偵測
4. 比較三種方法的敏感度 vs 延遲 tradeoff

### 前置條件

Lab 01 完成（`outputs/self-study/features.csv` 存在）

## 設計主線：偵測慢性偏移，而不是只追尖峰

本章的實務連結是「系統逐漸變壞」的情境。單點 threshold 擅長抓突發尖峰，但設備老化、queue 慢慢堆積、流量模式轉移常常不是單一時間點爆掉。

設計 SPC 規則時請問三個問題：

1. **正常期是否可信**：control limits 如果用已經異常的期間估計，後續偵測會被污染。
2. **記憶長度是否合理**：EWMA 越重視近期越敏感，但也更容易被噪音帶動。
3. **規則是否可解釋**：NOC 工程師能否理解這個 violation 代表什麼行動？

設計原則：SPC 是流程監控，不只是數學圖。它適合捕捉 drift、shift 與持續偏移。


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

def show_fig(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "environment.yml").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_SYNTHETIC = PROJECT_ROOT / "data" / "synthetic"
DATA_PROCESSED = PROJECT_ROOT / "outputs" / "self-study"
REPORTS = PROJECT_ROOT / "reports"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
features = pd.read_csv(DATA_PROCESSED / "features.csv", parse_dates=["timestamp"])
event_catalog = pd.read_csv(DATA_SYNTHETIC / "synthetic_event_catalog.csv", parse_dates=["start_time", "end_time"])
display(features[["timestamp", "port_id", "traffic_total", "discard_rate", "error_rate", "event_label"]].head())

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明約 8 分鐘，請先不要執行 cell。

---

## 📖 概念：Shewhart 管制圖與 3σ 邏輯

### 歷史背景

Walter Shewhart 在 1920 年代為貝爾電話公司設計了管制圖，
目的是區分「製程固有的隨機波動」和「需要介入的異常波動」。
這個想法在 100 年後仍然適用於網路監控，因為問題本質一樣：
「正常的波動」和「真正的問題」如何區分？

### 兩種變異的哲學

Shewhart 提出了根本性的區分：

| 類型 | 名稱 | 意義 | 正確回應 |
|---|---|---|---|
| 隨機波動 | Common cause variation | 製程本身的固有噪音，無法完全消除 | 接受它；調整閾值而非干預 |
| 異常偏差 | Special cause variation | 有具體原因的偏差，可以找到並修正 | 調查根因、介入處理 |

管制圖的工作：把這兩類分開。

### 3σ 管制線的邏輯

在穩定製程（常態分布）下：
- 99.73% 的點落在 μ ± 3σ 之內
- 出現 3σ 之外的機率 = 0.27%（每 370 個點才出現一次）

**Western Electric Rules**：Bell Labs 延伸了 Shewhart 的想法，
除了單點超出 3σ，還定義了幾種「連續型態也值得警覺」的規則：

| 規則 | 描述 | 捕捉的型態 |
|---|---|---|
| Rule 1 | 1 點在 ±3σ 之外 | 突發單點異常 |
| Rule 2 | 連續 9 點在中心線同側 | mean漂移（緩慢趨勢） |
| Rule 3 | 連續 6 點單調遞增或遞減 | 系統性趨勢（設備老化？） |
| Rule 4 | 連續 2 點在同側 2σ 以外 | 異常聚集 |

### 為什麼「用正常期間估計」control limits？

和 rolling z-score 不同，SPC 的 control limits 是從一段「已知正常的基準期」計算的，
不是從滑動窗口計算。這有一個重要含義：**基準期的選擇決定了你的「正常」定義**。
選到包含事件的基準期，控制線就被拉寬；選到太安靜的凌晨，
白天正常的流量峰值也會觸發告警。

**適用條件**：SPC 最適合製程有穩定基準（Baseline 可以確定）且變化主要是mean偏移的情況。
對於有強烈日 / 週季節性的網路流量，需要先去除季節性，否則 control limits 意義不大。

## Step 1 - 以正常期間估計 control limits

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

In [ ]:
rows = []
for port_id, g in features.groupby("port_id", sort=False):
    g = g.sort_values("timestamp").copy()
    base = g[g["event_label"] == "normal"].head(7 * 24 * 12)
    traffic_mu = base["traffic_total"].mean()
    traffic_sigma = base["traffic_total"].std()
    discard_mu = base["discard_rate"].mean()
    discard_sigma = base["discard_rate"].std()
    error_mu = base["error_rate"].mean()
    error_sigma = base["error_rate"].std()

    g["traffic_center"] = traffic_mu
    g["traffic_ucl"] = traffic_mu + 3 * traffic_sigma
    g["traffic_lcl"] = max(0, traffic_mu - 3 * traffic_sigma)
    g["shewhart_traffic_violation"] = g["traffic_total"] > g["traffic_ucl"]

    lam = 0.25
    g["ewma_discard_rate"] = g["discard_rate"].ewm(alpha=lam, adjust=False).mean()
    g["ewma_discard_ucl"] = discard_mu + 3 * discard_sigma * np.sqrt(lam / (2 - lam))
    g["ewma_discard_violation"] = g["ewma_discard_rate"] > g["ewma_discard_ucl"]

    k = 0.5 * max(error_sigma, 1e-12)
    h = 5 * max(error_sigma, 1e-12)
    s = 0.0
    pos = []
    for value in g["error_rate"].fillna(0):
        s = max(0, s + value - error_mu - k)
        pos.append(s)
    g["cusum_error_pos"] = pos
    g["cusum_error_h"] = h
    g["cusum_error_violation"] = g["cusum_error_pos"] > h
    rows.append(g)

spc = pd.concat(rows, ignore_index=True)
spc["any_spc_violation"] = spc[["shewhart_traffic_violation", "ewma_discard_violation", "cusum_error_violation"]].any(axis=1)
display(spc.loc[spc["any_spc_violation"], ["timestamp", "port_id", "event_label", "shewhart_traffic_violation", "ewma_discard_violation", "cusum_error_violation"]].head(12))

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 2 - 視覺化：Shewhart control chart

In [ ]:
port_id = "port-id7427"
p = spc[spc["port_id"] == port_id].copy()
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(p["timestamp"], p["traffic_total"], label="traffic_total", linewidth=1)
ax.plot(p["timestamp"], p["traffic_center"], label="center", linewidth=1.1)
ax.plot(p["timestamp"], p["traffic_ucl"], label="UCL", color="tab:red", linewidth=1.1)
ax.plot(p["timestamp"], p["traffic_lcl"], label="LCL", color="tab:green", linewidth=1.1)
v = p[p["shewhart_traffic_violation"]]
ax.scatter(v["timestamp"], v["traffic_total"], color="tab:red", s=18, label="violation")
for _, event in event_catalog.iterrows():
    if event["port_id"] in [port_id, "MULTI"]:
        ax.axvspan(event["start_time"], event["end_time"], alpha=0.10, color="tab:red")
ax.set_title(f"{port_id} - Shewhart chart for traffic_total")
ax.legend(loc="upper left")
ax.grid(alpha=0.25)
plt.tight_layout()
show_fig(fig)

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明約 7 分鐘，請先不要執行 cell。

---

## 📖 概念：EWMA — 有記憶衰退的移動平均

### 普通移動平均的問題

普通 rolling mean 把過去 N 個點當作同等重要。
但 5 分鐘前的流量，不是應該比 1 小時前的更重要嗎？

EWMA 的解法：**越近的資料，權重越大；越舊的資料，影響力逐漸消退。**

```
時間點：  t-5   t-4   t-3   t-2   t-1   t（current）
權重：    ░░░   ░░░   ░░░░  ████  ████  ████████
          最小                              最大
          (舊的記憶逐漸淡化)         (新資料最重要)
```

λ（平滑參數）控制記憶長度：

| λ 值 | 記憶長度 | 特性 | 適合偵測 |
|---|---|---|---|
| 大（0.5）| 短（偏重近期）| 反應快，對噪音敏感 | 快速變化 |
| 小（0.1）| 長（加權平滑）| 反應慢，過濾雜訊 | 緩慢漂移 |

📎 延伸閱讀：[EWMA Chart — Wikipedia](https://en.wikipedia.org/wiki/EWMA_chart)（有控制圖示意）

---

## 📖 概念：CUSUM — 偏差的累積偵測器

### 核心思想

CUSUM 不看當下的值，而是**追蹤「小偏差是否在持續累積」**。

```
情境：流量每個點都比正常高出一點點（+2%）

Shewhart 每次看單點 → 沒有超出 3σ → 不觸發
CUSUM 累積偏差 →  +2% → +4% → +6% → +8% → ... → 超過閾值 → 觸發

```

**直觀類比**：就像裁判不記得每一次小犯規，
但如果同一個動作一直重複，終究會被叫停。

### 三種方法的適用場景

```
異常型態示意

突發尖峰：      _______↑↑↑↑↑_______    ← Shewhart 最好用
                              （幾個點就完了）

持續漂移：      ____________/─────    ← CUSUM 最好用
                              （緩慢但持續偏高）

緩慢趨勢：      ___________/─────    ← EWMA（小 λ）最好用
                              （移動平均跟著上升）
```

📎 延伸閱讀：[CUSUM — Wikipedia](https://en.wikipedia.org/wiki/CUSUM)（有圖）


## Step 3 - 視覺化：EWMA and CUSUM

EWMA 適合小幅持續偏移，CUSUM 適合累積性變化。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(p["timestamp"], p["discard_rate"], alpha=0.35, label="discard_rate")
axes[0].plot(p["timestamp"], p["ewma_discard_rate"], label="EWMA discard_rate")
axes[0].plot(p["timestamp"], p["ewma_discard_ucl"], label="EWMA UCL", color="tab:red")
axes[0].set_title("EWMA chart for discard_rate")
axes[0].legend(loc="upper left")

axes[1].plot(p["timestamp"], p["cusum_error_pos"], label="CUSUM error positive")
axes[1].plot(p["timestamp"], p["cusum_error_h"], label="decision interval h", color="tab:red")
axes[1].set_title("CUSUM chart for error_rate")
axes[1].legend(loc="upper left")

for ax in axes:
    for _, event in event_catalog.iterrows():
        if event["port_id"] in [port_id, "MULTI"]:
            ax.axvspan(event["start_time"], event["end_time"], alpha=0.10, color="tab:red")
    ax.grid(alpha=0.25)
plt.tight_layout()
show_fig(fig)

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 4 - SPC violation summary 與輸出

In [ ]:
summary = (
    spc.groupby("event_label")
    .agg(rows=("timestamp", "size"), spc_violations=("any_spc_violation", "sum"))
    .assign(violation_rate=lambda x: x["spc_violations"] / x["rows"])
    .sort_values("violation_rate", ascending=False)
)
display(summary)

keep = [
    "timestamp", "device_id", "port_id", "port_role", "event_label", "event_id",
    "traffic_total", "traffic_center", "traffic_ucl", "traffic_lcl", "shewhart_traffic_violation",
    "discard_rate", "ewma_discard_rate", "ewma_discard_ucl", "ewma_discard_violation",
    "error_rate", "cusum_error_pos", "cusum_error_h", "cusum_error_violation", "any_spc_violation",
]
out_path = DATA_PROCESSED / "spc_results.csv"
spc[keep].to_csv(out_path, index=False)
print(f"saved: {out_path}")

---

## ⚖️ 方法比較：SPC vs EWMA vs CUSUM

| | Shewhart (3σ) | EWMA | CUSUM |
|---|---|---|---|
| **核心思想** | 單點超出固定界線 | 指數衰減平均的界線 | 偏差累積計數 |
| **最佳場景** | 突發尖峰（DDoS、硬體故障） | 緩慢漂移（設備老化、配置錯誤） | 持續微小偏移（記憶體洩漏、連線池耗盡） |
| **延遲** | 最低（即時） | 中 | 中到高 |
| **對噪音的敏感度** | 高（容易誤報） | 低（λ 小時更平滑） | 低（k 值過濾噪音） |
| **參數** | σ 倍數 | λ（平滑係數） | k（參考值）、h（閾值） |
| **前置假設** | 資料接近常態 | 資料穩態時有固定 mean | 基準 mean 已知且穩定 |

### 本課程的選擇原則

網路事件的多樣性決定了沒有單一最佳方法：
- `error_burst`（突然大量錯誤）→ Shewhart Rule 1 最快偵測到
- `traffic_drop`（連線逐漸下降，可能是路由問題）→ CUSUM 最敏感
- `queue_congestion`（流量持續超出容量）→ EWMA 或 CUSUM 都有效

實務建議：先跑 Shewhart，把未觸發的部分再送 CUSUM，可以降低整體延遲並提高召回率。

---
🔧 **探索練習** — 修改指定參數，觀察結果變化。

---

## 🔧 探索練習：EWMA λ 值的影響

在 Step 3 的 code cell 中找到 EWMA 的 λ 參數，從 `0.2` 改成 `0.5`，
重新執行並觀察 EWMA 曲線的平滑程度變化。

- λ = 0.5 時，EWMA 偵測到事件的速度是否比 λ = 0.2 更快？
- 但在正常時段，λ = 0.5 是否出現更多「假觸發」？

這個練習展示了 sensitivity vs lag 的基本 tradeoff。

---

> 「如果你的基準期（正常期間）選得太短，control limits 會發生什麼？」

> 「CUSUM 偵測到告警後，你要怎麼知道它從什麼時候「開始積累」？這對 RCA 有什麼幫助？」

> 「Week-of-day 的效應（週末流量低、平日流量高）對 Shewhart control limits 有什麼影響？如何處理？」

## ⚠ 人類驗證點 #3 — 管制規則是流程決策，不是數學答案

SPC 管制圖的 3σ 界線是理論基礎，但「用哪條 Western Electric 規則」是業務決策。

### 如何判斷

選好管制規則後，問自己：

- 在已知事件窗口（events.csv）內，規則是否觸發了？（Recall）
- 在正常時段，每天大約觸發幾次？（False positive rate）
- 運維人員接到這些告警後，有多少是有意義的？

### 讓你重新考慮的信號

- UCL / LCL 在正常時段每天觸發超過 3 次 → 管制線可能設得太嚴
- 已知事件完全未觸發 → 選用的規則可能不適合這種異常類型
- CUSUM 偵測到 Shewhart 沒偵測到的漂移 → 考慮兩者並用

### 🔧 探索練習

在 Step 3 的 code cell 找到以下參數，分別修改並重新執行：

```python
lambda_ewma = 0.3   # 改成 0.1（更平滑）或 0.5（更敏感），觀察管制線的變化
k_cusum = 0.5       # 改成 0.3（更敏感）或 1.0（更保守），觀察累積偏差的觸發時機
```

> 「Shewhart 和 CUSUM 偵測到的是同一個事件嗎？哪個先偵測到？Time差多少？」

> 「你的網路環境裡，哪種異常更常見：突發尖峰，還是緩慢漂移？這個答案會影響你選哪個方法。」

> 「如果同時用三種方法，你會怎麼合併它們的輸出？多數決？優先級？」
